In [ ]:
# ============================================================
# Database design in phpMyAdmin -> SQL
# ============================================================

In [ ]:
# ============================================================
# Import_runs table -> SQL
# ============================================================

In [1]:
# ============================================================
# Phone normalization
# ============================================================
import re


def normalize_iran_phone(value: str) -> str:
    """
    Return Iranian mobile number in local 10-digit form:
        9121234567

    Accepted examples:
        9121234567
        09121234567
        +989121234567
        00989121234567
    """

    if value is None:
        raise ValueError("Phone number is empty")

    s = str(value).strip()

    # Remove spaces, -, parentheses, etc.
    s = re.sub(r"[^\d+]", "", s)

    if s.startswith("+98"):
        s = s[3:]
    elif s.startswith("0098"):
        s = s[4:]
    elif s.startswith("98") and len(s) == 12:
        s = s[2:]
    elif s.startswith("0"):
        s = s[1:]

    if not re.fullmatch(r"9\d{9}", s):
        raise ValueError(f"Invalid Iranian mobile number: {value!r}")

    return s

In [2]:
# ============================================================
# Import Excel into the database
# ============================================================
from pathlib import Path

import pandas as pd
import mysql.connector

EXCEL_FILE = Path(
    "/home/saeed/repositories/contacts/import_eitaa/contacts-1-44.xlsx"
)


def get_db_connection():
    return mysql.connector.connect(
        host="127.0.0.1",
        port=3306,
        user="reza",
        password="123",
        database="eitaa_import",
        charset="utf8mb4",
    )


def import_excel():
    df = pd.read_excel(
        EXCEL_FILE,
        dtype={"Full Name": str, "Phone Number": str},
    )

    expected = {"Full Name", "Phone Number"}

    if set(df.columns) != expected:
        raise ValueError(
            f"Expected columns {expected}, got {set(df.columns)}"
        )

    conn = get_db_connection()

    try:
        cursor = conn.cursor()

        sql = """
            INSERT INTO contacts (
                full_name,
                phone_original,
                phone_normalized,
                status
            )
            VALUES (%s, %s, %s, 'pending')
            ON DUPLICATE KEY UPDATE
                full_name = VALUES(full_name),
                phone_original = VALUES(phone_original)
        """

        inserted = 0

        for _, row in df.iterrows():
            full_name = str(row["Full Name"]).strip()
            phone_original = str(row["Phone Number"]).strip()

            if not full_name:
                continue

            try:
                phone_normalized = normalize_iran_phone(
                    phone_original
                )
            except ValueError as exc:
                print(
                    f"Invalid phone number "
                    f"{phone_original!r}: {exc}"
                )
                continue

            cursor.execute(
                sql,
                (
                    full_name,
                    phone_original,
                    phone_normalized,
                ),
            )

            inserted += 1

        conn.commit()

        print(f"Imported/updated {inserted} contacts.")

    finally:
        cursor.close()
        conn.close()


if __name__ == "__main__":
    import_excel()

Imported/updated 74 contacts.


In [3]:
# ============================================================
# Chrome session persistence
# ============================================================
from selenium import webdriver
from selenium.webdriver.chrome.options import Options

CHROME_PROFILE = "/home/saeed/.config/eitaa-chrome"


def create_driver():
    options = Options()

    options.add_argument(
        f"--user-data-dir={CHROME_PROFILE}"
    )

    options.add_argument("--start-maximized")

    driver = webdriver.Chrome(options=options)

    driver.set_page_load_timeout(60)

    return driver

In [4]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service

CHROMEDRIVER = (
    Path.cwd()
    / "drivers"
    / "chromedriver"
    / "chromedriver"
)

assert CHROMEDRIVER.is_file(), f"ChromeDriver not found: {CHROMEDRIVER}"

driver = webdriver.Chrome(
    service=Service(executable_path=str(CHROMEDRIVER))
)

driver.get("https://www.google.com")

First run:

driver = create_driver()

driver.get("https://web.eitaa.com/")

Log into Eitaa manually.

Close the browser normally.

On subsequent executions Selenium will reuse that Chrome profile/session.

This is much more robust than trying to automate OTP authentication.

In [5]:
# ============================================================
# Put selectors in one place
# ============================================================
# To test that the selector is unique:
# allow pasting
# document.querySelectorAll(".btn-menu-item.tgico-user.rp").length

from selenium.webdriver.common.by import By


class Selectors:
    # 1. Hamburger menu
    HAMBURGER_MENU = (
        By.CSS_SELECTOR,
        ".btn-icon.btn-menu-toggle.rp.sidebar-tools-button.is-visible"
    )

    # 2. Contacts menu item
    CONTACTS_BUTTON = (
        By.CSS_SELECTOR,
        ".btn-menu-item.tgico-user.rp"
    )

    # 3. "+" button inside Contacts
    ADD_CONTACT_BUTTON = (
        By.CSS_SELECTOR,
        "button.tgico-add"
    )

    # 4. First name field
    FIRST_NAME = (
        By.CSS_SELECTOR,
        ".name-fields > div:nth-child(1)"
    )

    # Phone number field
    PHONE = (
        By.CSS_SELECTOR,
        'div.input-field-phone > div.input-field-input[contenteditable="true"]'
    )

    # Add/Save button
    SAVE_BUTTON = (
        By.CSS_SELECTOR,
        ".popup-create-contact button.btn-primary.btn-color-primary"
    )

In [6]:
# ============================================================
# Reliable waits
# ============================================================
# Avoid: time.sleep(3) as the primary synchronization mechanism.
# Use WebDriverWait.

from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

WAIT_SECONDS = 30


def wait_visible(driver, locator):
    return WebDriverWait(
        driver,
        WAIT_SECONDS
    ).until(
        EC.visibility_of_element_located(locator)
    )


def wait_clickable(driver, locator):
    return WebDriverWait(
        driver,
        WAIT_SECONDS
    ).until(
        EC.element_to_be_clickable(locator)
    )


In [7]:
# ============================================================
# The actual Eitaa contact operation
# ============================================================
def add_contact(driver, full_name, phone_normalized):
    """
    Add exactly one Eitaa contact.

    Full name intentionally goes entirely into
    the first-name field.
    """

    # Open hamburger menu.
    wait_clickable(
        driver,
        Selectors.HAMBURGER_MENU
    ).click()

    # Open contacts.
    wait_clickable(
        driver,
        Selectors.CONTACTS_BUTTON
    ).click()

    # Open Add Contact.
    wait_clickable(
        driver,
        Selectors.ADD_CONTACT_BUTTON
    ).click()

    # First name = COMPLETE full name.
    first_name = wait_visible(
        driver,
        Selectors.FIRST_NAME
    )

    first_name.clear()
    first_name.send_keys(full_name)

    # Eitaa already displays +98.
    phone = wait_visible(
        driver,
        Selectors.PHONE
    )

    phone.clear()
    phone.send_keys(phone_normalized)

    # Submit.
    wait_clickable(
        driver,
        Selectors.SAVE_BUTTON
    ).click()

    # IMPORTANT:
    # Wait for confirmation/state change here.
    #
    # Replace this with a selector for whatever
    # Eitaa actually displays after successful
    # contact creation.

In [8]:
# ============================================================
# Fetch the next contact
# ============================================================
def get_next_contact(conn):
    cursor = conn.cursor(dictionary=True)

    try:
        cursor.execute("""
            SELECT
                id,
                full_name,
                phone_original,
                phone_normalized,
                attempts
            FROM contacts
            WHERE status IN ('pending', 'failed')
              AND attempts < 3
            ORDER BY id
            LIMIT 1
        """)

        return cursor.fetchone()

    finally:
        cursor.close()

# You may want failed contacts handled separately rather than automatically retried. For 5,000 records, I generally recommend:

# pending → automatic retry
# failed  → max 2-3 attempts

# and then review the failures manually.

In [9]:
# ============================================================
# Complete processing loop
# ============================================================
# Here is the basic architecture I would use:

import logging
from datetime import datetime

from selenium.common.exceptions import (
    TimeoutException,
    WebDriverException,
)

logging.basicConfig(
    filename="eitaa_import.log",
    level=logging.INFO,
    format=(
        "%(asctime)s | %(levelname)s | "
        "%(message)s"
    ),
)


def mark_processing(conn, contact_id):
    cursor = conn.cursor()

    try:
        cursor.execute("""
            UPDATE contacts
            SET
                status = 'processing',
                attempts = attempts + 1,
                started_at = NOW()
            WHERE id = %s
        """, (contact_id,))

        conn.commit()

    finally:
        cursor.close()


def mark_success(conn, contact_id):
    cursor = conn.cursor()

    try:
        cursor.execute("""
            UPDATE contacts
            SET
                status = 'success',
                error_message = NULL,
                completed_at = NOW()
            WHERE id = %s
        """, (contact_id,))

        conn.commit()

    finally:
        cursor.close()


def mark_failed(conn, contact_id, error):
    cursor = conn.cursor()

    try:
        cursor.execute("""
            UPDATE contacts
            SET
                status = 'failed',
                error_message = %s
            WHERE id = %s
        """, (
            str(error)[:60000],
            contact_id,
        ))

        conn.commit()

    finally:
        cursor.close()


def reset_stale_processing(conn):
    cursor = conn.cursor()

    try:
        cursor.execute("""
            UPDATE contacts
            SET status = 'pending'
            WHERE status = 'processing'
              AND started_at <
                  NOW() - INTERVAL 30 MINUTE
        """)

        conn.commit()

    finally:
        cursor.close()


def run_import():
    conn = get_db_connection()

    driver = None

    try:
        reset_stale_processing(conn)

        driver = create_driver()

        driver.get("https://web.eitaa.com/")

        # At this point, verify that the existing
        # Eitaa session is still authenticated.
        #
        # If not, stop and ask for manual login
        # rather than attempting OTP automatically.

        while True:

            contact = get_next_contact(conn)

            if contact is None:
                logging.info("No more contacts.")
                break

            contact_id = contact["id"]
            full_name = contact["full_name"]
            phone = contact["phone_normalized"]

            logging.info(
                "Processing id=%s name=%r phone=%s",
                contact_id,
                full_name,
                phone,
            )

            mark_processing(conn, contact_id)

            try:

                add_contact(
                    driver,
                    full_name,
                    phone,
                )

                mark_success(
                    conn,
                    contact_id,
                )

                logging.info(
                    "SUCCESS id=%s",
                    contact_id,
                )

            except TimeoutException as exc:

                mark_failed(
                    conn,
                    contact_id,
                    f"Timeout: {exc}",
                )

                logging.exception(
                    "TIMEOUT id=%s",
                    contact_id,
                )

            except WebDriverException as exc:

                mark_failed(
                    conn,
                    contact_id,
                    f"WebDriver error: {exc}",
                )

                logging.exception(
                    "WEBDRIVER ERROR id=%s",
                    contact_id,
                )

                # Browser may be unusable.
                # Recreate it.

                try:
                    driver.quit()
                except Exception:
                    pass

                driver = create_driver()
                driver.get(
                    "https://web.eitaa.com/"
                )

            except Exception as exc:

                mark_failed(
                    conn,
                    contact_id,
                    repr(exc),
                )

                logging.exception(
                    "ERROR id=%s",
                    contact_id,
                )

    finally:

        if driver is not None:
            try:
                driver.quit()
            except Exception:
                pass

        conn.close()

In [ ]:
# ============================================================
# Chrome crash recovery
# ============================================================
except WebDriverException:

    logging.exception(
        "Chrome/WebDriver failure"
    )

    try:
        driver.quit()
    except Exception:
        pass

    driver = create_driver()

    driver.get(
        "https://web.eitaa.com/"
    )

    # Wait for authenticated Eitaa state again.

In [ ]:
# ============================================================
# Add screenshots for failures
# ============================================================
from pathlib import Path
from datetime import datetime


SCREENSHOT_DIR = Path("screenshots")
SCREENSHOT_DIR.mkdir(exist_ok=True)


def save_failure_screenshot(driver, contact_id):
    filename = (
        SCREENSHOT_DIR
        / f"{contact_id}_{datetime.now():%Y%m%d_%H%M%S}.png"
    )

    driver.save_screenshot(str(filename))

    return filename




In [ ]:
except Exception as exc:

    screenshot = save_failure_screenshot(
        driver,
        contact_id,
    )

    mark_failed(
        conn,
        contact_id,
        f"{exc}\nScreenshot: {screenshot}",
    )